# QuizForge – Question Generation Demo (no UI)
This notebook runs the same pipeline as the Streamlit app, step by step, so each NLP stage can be inspected.

**Stages:** text extraction → cleaning → chunking → Seq2Seq LLM (Flan-T5) question generation → answer → TF-IDF keywords.

Run it from the `notebooks/` folder of the project (or in Colab after uploading the whole project folder).

## 1. Install requirements

In [ ]:
!pip install -q -r ../requirements.txt

## 2. Imports and project path

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))   # so `import qgen` works from notebooks/

from qgen.extractor import extract_text, clean_text, build_chunks, select_chunks
from qgen.generator import QuestionGenerator, MODEL_OPTIONS
from qgen.pipeline import generate_quiz

## 3. Load a textbook (PDF or DOCX)
Set `BOOK_PATH` to your own file (keep it under 5 MB, same as the app).

In [ ]:
BOOK_PATH = "my_textbook.pdf"   # <-- change this

with open(BOOK_PATH, "rb") as f:
    raw_text = extract_text(BOOK_PATH, f.read())
print(len(raw_text.split()), "words extracted")
print(raw_text[:500])

## 4. Clean and chunk the text

In [ ]:
text = clean_text(raw_text)
chunks = build_chunks(text)
print(len(chunks), "usable passages")
print("\nExample passage:\n", chunks[0])

## 5. Load the Seq2Seq Transformer LLM
The first run downloads the weights from Hugging Face.

In [ ]:
generator = QuestionGenerator("google/flan-t5-base")
print("Running on:", generator.device)

## 6. Generate questions
Choose `difficulty` = `low` / `mid` / `high` and `NUM_QUESTIONS` = multiple of 5 (max 30).

In [ ]:
DIFFICULTY = "mid"
NUM_QUESTIONS = 5

items = generate_quiz(text, generator, DIFFICULTY, NUM_QUESTIONS,
                      progress_callback=lambda p: print(f"\rprogress: {p:.0%}", end=""))
print()
for i, it in enumerate(items, 1):
    print(f"Q{i}. {it.question}")

## 7. Answers and keywords

In [ ]:
for i, it in enumerate(items, 1):
    print(f"Q{i}. {it.question}")
    print("   Keywords:", ", ".join(it.keywords))
    print("   Answer  :", it.answer, "\n")

## 8. (Optional) Compare difficulty levels on one passage
Shows how only the *prompt* changes the question style.

In [ ]:
passage = chunks[0]
for level in ["low", "mid", "high"]:
    print(level.upper(), "->", generator.generate_question(passage, level))